# 模型导出教程 (Model Export Tutorial)

> **前置知识**: PyTorch 基础、深度学习模型训练流程
>
> **学习目标**: 掌握模型导出的原理、格式选择和最佳实践

---

## 为什么需要模型导出？

```
┌─────────────────────────────────────────────────────────────┐
│                   模型导出的核心价值                         │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  训练环境                          部署环境                 │
│  ┌─────────────────┐              ┌─────────────────┐       │
│  │ Python + PyTorch│              │ C++ / Java / Go │       │
│  │ GPU 服务器      │    导出      │ 边缘设备/手机   │       │
│  │ 动态图模式      │ ──────────→  │ 静态图优化      │       │
│  │ 调试友好        │              │ 高性能推理      │       │
│  └─────────────────┘              └─────────────────┘       │
│                                                             │
│  导出格式选择:                                              │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  TorchScript: PyTorch 原生，支持 Python 子集        │   │
│  │  ONNX: 跨框架通用，生态丰富                         │   │
│  │  TensorRT: NVIDIA GPU 极致优化                      │   │
│  │  CoreML: Apple 设备专用                             │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

## 本教程内容

1. **模型分析** - 导出前的参数统计和性能分析
2. **TorchScript 导出** - Tracing vs Scripting
3. **ONNX 导出** - 跨框架通用格式
4. **导出验证** - 确保导出正确性
5. **性能对比** - 不同格式的推理性能

In [ ]:
# ============================================================
# 环境准备
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import tempfile
import os
import time
import numpy as np

# 设置随机种子，确保结果可复现
torch.manual_seed(42)
np.random.seed(42)

print("=" * 50)
print("环境准备完成")
print("=" * 50)
print(f"PyTorch 版本: {torch.__version__}")

<cell_type>markdown</cell_type>## 1. 定义测试模型

**核心概念**: 在导出前，需要一个训练好的模型作为示例

```
┌─────────────────────────────────────────────────────────────┐
│                    测试模型架构                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  输入: [B, 3, 32, 32]                                       │
│         ↓                                                   │
│  Conv1 (3→32) + BN + ReLU + Pool                           │
│         ↓ [B, 32, 16, 16]                                   │
│  Conv2 (32→64) + BN + ReLU + Pool                          │
│         ↓ [B, 64, 8, 8]                                     │
│  Flatten                                                    │
│         ↓ [B, 4096]                                         │
│  FC1 (4096→256) + ReLU + Dropout                           │
│         ↓ [B, 256]                                          │
│  FC2 (256→10)                                               │
│         ↓ [B, 10]                                           │
│  输出: 10 类分类 logits                                     │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 定义测试模型
# ============================================================

class ImageClassifier(nn.Module):
    """
    图像分类模型
    
    一个简单的 CNN 模型，用于演示模型导出
    
    结构:
    - 2 个卷积层 (带 BatchNorm)
    - 2 个全连接层
    - Dropout 正则化
    """
    def __init__(self, num_classes=10):
        super().__init__()
        # 卷积层 1: 3 通道 → 32 通道
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        
        # 卷积层 2: 32 通道 → 64 通道
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        
        # 池化层
        self.pool = nn.MaxPool2d(2)
        
        # 全连接层
        self.fc1 = nn.Linear(64 * 8 * 8, 256)  # 64通道 × 8×8 特征图
        self.fc2 = nn.Linear(256, num_classes)
        
        # Dropout
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        # 卷积块 1: Conv → BN → ReLU → Pool
        x = self.pool(F.relu(self.bn1(self.conv1(x))))  # [B,3,32,32] → [B,32,16,16]
        
        # 卷积块 2: Conv → BN → ReLU → Pool
        x = self.pool(F.relu(self.bn2(self.conv2(x))))  # [B,32,16,16] → [B,64,8,8]
        
        # 展平
        x = x.view(x.size(0), -1)  # [B,64,8,8] → [B,4096]
        
        # 全连接层
        x = self.dropout(F.relu(self.fc1(x)))  # [B,4096] → [B,256]
        return self.fc2(x)  # [B,256] → [B,10]


# 创建模型并设置为评估模式
model = ImageClassifier()
model.eval()  # 重要: 导出前必须设置为 eval 模式!

# 创建示例输入
dummy_input = torch.randn(1, 3, 32, 32)

# 测试前向传播
print("=" * 60)
print("测试模型")
print("=" * 60)
with torch.no_grad():
    output = model(dummy_input)
print(f"输入形状: {dummy_input.shape}")
print(f"输出形状: {output.shape}")
print(f"输出示例: {output[0, :5].numpy().round(3)}...")

<cell_type>markdown</cell_type>## 2. 模型分析

**核心概念**: 在导出前，先分析模型的结构、参数量和性能

```
┌─────────────────────────────────────────────────────────────┐
│                   导出前分析清单                             │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  1. 参数统计                                                │
│     ├── 总参数量                                           │
│     ├── 可训练参数                                         │
│     └── 各层参数分布                                       │
│                                                             │
│  2. 模型大小估算                                            │
│     ├── FP32: 4 字节/参数                                  │
│     ├── FP16: 2 字节/参数                                  │
│     └── INT8: 1 字节/参数                                  │
│                                                             │
│  3. 推理性能                                                │
│     ├── 平均延迟                                           │
│     ├── 吞吐量                                             │
│     └── 内存占用                                           │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 模型分析工具
# ============================================================

def count_parameters(model):
    """
    统计模型参数量
    
    返回:
        dict: 包含总参数量、可训练参数量、各层参数量
    """
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    # 各层参数量
    layers = {}
    for name, param in model.named_parameters():
        layers[name] = param.numel()
    
    return {
        'total': total,
        'trainable': trainable,
        'layers': layers
    }


def estimate_model_size(model, dtype=torch.float32):
    """
    估算模型大小
    
    参数:
        model: PyTorch 模型
        dtype: 数据类型 (float32, float16, int8)
        
    返回:
        dict: 包含参数大小和缓冲区大小
    """
    # 每种数据类型的字节数
    bytes_per_element = {
        torch.float32: 4,
        torch.float16: 2,
        torch.int8: 1,
        torch.bfloat16: 2
    }
    
    element_size = bytes_per_element.get(dtype, 4)
    
    # 参数大小
    param_size = sum(p.numel() for p in model.parameters()) * element_size
    
    # 缓冲区大小 (如 BatchNorm 的 running_mean)
    buffer_size = sum(b.numel() * b.element_size() for b in model.buffers())
    
    total_size = param_size + buffer_size
    
    return {
        'param_bytes': param_size,
        'buffer_bytes': buffer_size,
        'total_bytes': total_size,
        'total_mb': total_size / (1024 * 1024)
    }


# ============================================================
# 参数统计
# ============================================================
print("=" * 60)
print("模型参数统计")
print("=" * 60)

param_stats = count_parameters(model)

print(f"\n总参数量: {param_stats['total']:,}")
print(f"可训练参数: {param_stats['trainable']:,}")
print(f"\n各层参数量:")
for name, count in param_stats['layers'].items():
    print(f"  {name}: {count:,}")

In [ ]:
# ============================================================
# 模型大小估算
# ============================================================
print("=" * 60)
print("模型大小估算")
print("=" * 60)

size_fp32 = estimate_model_size(model, torch.float32)
size_fp16 = estimate_model_size(model, torch.float16)
size_int8 = estimate_model_size(model, torch.int8)

print(f"\n不同精度下的模型大小:")
print(f"  FP32: {size_fp32['total_mb']:.2f} MB")
print(f"  FP16: {size_fp16['total_mb']:.2f} MB (压缩 2x)")
print(f"  INT8: {size_int8['total_mb']:.2f} MB (压缩 4x)")

In [ ]:
# ============================================================
# 推理性能分析
# ============================================================

def profile_inference(model, input_shape, num_runs=50, warmup_runs=10, device="cpu"):
    """
    分析模型推理性能
    
    参数:
        model: PyTorch 模型
        input_shape: 输入形状
        num_runs: 测试次数
        warmup_runs: 预热次数
        device: 设备
        
    返回:
        dict: 包含延迟统计和吞吐量
    """
    model = model.to(device)
    model.eval()
    
    dummy_input = torch.randn(*input_shape).to(device)
    
    # 预热 (让 CPU/GPU 缓存稳定)
    with torch.no_grad():
        for _ in range(warmup_runs):
            _ = model(dummy_input)
    
    # 计时
    times = []
    with torch.no_grad():
        for _ in range(num_runs):
            start = time.perf_counter()
            _ = model(dummy_input)
            end = time.perf_counter()
            times.append((end - start) * 1000)  # 转换为毫秒
    
    times = np.array(times)
    
    return {
        'mean_ms': times.mean(),
        'std_ms': times.std(),
        'min_ms': times.min(),
        'max_ms': times.max(),
        'throughput': 1000 / times.mean()  # samples/sec
    }


print("=" * 60)
print("推理性能分析")
print("=" * 60)

perf_stats = profile_inference(
    model,
    input_shape=(1, 3, 32, 32),
    num_runs=50,
    warmup_runs=10,
    device="cpu"
)

print(f"\n推理性能 (CPU, batch_size=1):")
print(f"  平均延迟: {perf_stats['mean_ms']:.2f} ± {perf_stats['std_ms']:.2f} ms")
print(f"  最小延迟: {perf_stats['min_ms']:.2f} ms")
print(f"  最大延迟: {perf_stats['max_ms']:.2f} ms")
print(f"  吞吐量: {perf_stats['throughput']:.1f} samples/sec")

<cell_type>markdown</cell_type>## 3. TorchScript 导出

**核心概念**: TorchScript 是 PyTorch 的原生序列化格式，将动态图转换为静态图

```
┌─────────────────────────────────────────────────────────────┐
│                   TorchScript 两种方式                       │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  1. Tracing (追踪)                                          │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  原理: 运行一次前向传播，记录执行路径               │   │
│  │  优点: 简单，不需要修改代码                         │   │
│  │  缺点: 不支持动态控制流 (if/for 依赖输入)           │   │
│  │  适用: 静态计算图的模型 (大多数 CNN)                │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  2. Scripting (脚本化)                                      │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  原理: 分析 Python 代码，转换为 TorchScript IR      │   │
│  │  优点: 支持动态控制流                               │   │
│  │  缺点: 需要类型注解，部分 Python 特性不支持         │   │
│  │  适用: 包含条件分支的模型                           │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  选择建议:                                                  │
│  - 模型有 if/for 依赖输入 → 用 script                      │
│  - 模型是纯前馈网络 → 用 trace (更简单)                    │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# TorchScript Tracing (追踪)
# ============================================================
print("=" * 60)
print("TorchScript Tracing (追踪)")
print("=" * 60)

# Tracing: 运行一次前向传播，记录执行路径
# 注意: 需要提供示例输入
traced_model = torch.jit.trace(model, dummy_input)

# 验证输出一致性
with torch.no_grad():
    original_out = model(dummy_input)
    traced_out = traced_model(dummy_input)

# 比较输出
is_close = torch.allclose(original_out, traced_out, atol=1e-5)
max_diff = (original_out - traced_out).abs().max().item()

print(f"\n验证结果:")
print(f"  输出一致: {is_close}")
print(f"  最大差异: {max_diff:.6f}")

# 查看追踪后的计算图
print(f"\n追踪后的计算图节点数: {len(list(traced_model.graph.nodes()))})")

In [ ]:
# ============================================================
# TorchScript Scripting (脚本化)
# ============================================================
print("=" * 60)
print("TorchScript Scripting (脚本化)")
print("=" * 60)

# Scripting: 分析 Python 代码，转换为 TorchScript IR
scripted_model = torch.jit.script(model)

# 验证输出一致性
with torch.no_grad():
    scripted_out = scripted_model(dummy_input)

is_close = torch.allclose(original_out, scripted_out, atol=1e-5)
max_diff = (original_out - scripted_out).abs().max().item()

print(f"\n验证结果:")
print(f"  输出一致: {is_close}")
print(f"  最大差异: {max_diff:.6f}")

In [ ]:
# ============================================================
# 查看 TorchScript 生成的代码
# ============================================================
print("=" * 60)
print("TorchScript 生成的代码")
print("=" * 60)

print("\nScripted 模型的 forward 方法:")
print(scripted_model.code)

In [ ]:
# ============================================================
# 保存和加载 TorchScript 模型
# ============================================================
print("=" * 60)
print("保存和加载 TorchScript 模型")
print("=" * 60)

with tempfile.TemporaryDirectory() as tmpdir:
    # 保存模型
    save_path = os.path.join(tmpdir, "model_traced.pt")
    traced_model.save(save_path)
    
    # 检查文件大小
    file_size = os.path.getsize(save_path) / (1024 * 1024)
    print(f"\n保存路径: {save_path}")
    print(f"文件大小: {file_size:.2f} MB")
    
    # 加载模型 (可以在没有原始模型定义的情况下加载!)
    loaded_model = torch.jit.load(save_path)
    
    # 验证加载后的模型
    with torch.no_grad():
        loaded_out = loaded_model(dummy_input)
    
    is_close = torch.allclose(original_out, loaded_out, atol=1e-5)
    print(f"\n加载后验证:")
    print(f"  输出一致: {is_close}")
    
print("\n✓ TorchScript 模型可以独立于 Python 代码运行!")

<cell_type>markdown</cell_type>### Tracing vs Scripting 对比

| 特性 | Tracing | Scripting |
|:-----|:--------|:----------|
| 原理 | 记录执行路径 | 分析 Python 代码 |
| 控制流 | 不支持动态 | 支持动态 |
| 实现难度 | 简单 | 需要类型注解 |
| 适用场景 | 静态模型 (CNN) | 动态模型 (RNN, Transformer) |

```
Tracing 的局限性:
┌─────────────────────────────────────────────────────────────┐
│  def forward(self, x, use_relu=True):                      │
│      x = self.fc(x)                                        │
│      if use_relu:  # ← 这个分支在 trace 时被固定!          │
│          x = F.relu(x)                                     │
│      return x                                              │
│                                                             │
│  # 如果 trace 时 use_relu=True，则永远执行 relu            │
│  # 即使推理时传入 use_relu=False 也会执行 relu!            │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 带控制流的模型示例
# ============================================================
print("=" * 60)
print("动态控制流示例")
print("=" * 60)

class DynamicModel(nn.Module):
    """
    带动态控制流的模型
    
    forward 方法中包含依赖输入的 if 语句
    这种模型必须使用 Scripting，不能使用 Tracing
    """
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 10)
    
    def forward(self, x, use_relu: bool = True):
        x = self.fc(x)
        if use_relu:  # 动态控制流: 依赖输入参数
            x = F.relu(x)
        return x


dynamic_model = DynamicModel()
dynamic_model.eval()

# Scripting 可以正确处理控制流
scripted_dynamic = torch.jit.script(dynamic_model)

x = torch.randn(1, 10)

print("\nScripting 支持动态控制流:")
with torch.no_grad():
    out_relu = scripted_dynamic(x, True)
    out_no_relu = scripted_dynamic(x, False)

print(f"  use_relu=True:  输出范围 [{out_relu.min():.3f}, {out_relu.max():.3f}]")
print(f"  use_relu=False: 输出范围 [{out_no_relu.min():.3f}, {out_no_relu.max():.3f}]")
print(f"\n  ✓ 两种模式输出不同，控制流正确工作!")

<cell_type>markdown</cell_type>## 4. ONNX 导出

**核心概念**: ONNX (Open Neural Network Exchange) 是跨框架的通用模型格式

```
┌─────────────────────────────────────────────────────────────┐
│                    ONNX 格式介绍                             │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  ONNX 是什么？                                              │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  跨框架的模型交换格式                               │   │
│  │                                                     │   │
│  │  PyTorch ─┐                                        │   │
│  │           │                                        │   │
│  │  TensorFlow ──→ ONNX ──→ ONNX Runtime             │   │
│  │           │              TensorRT                  │   │
│  │  其他框架 ─┘              OpenVINO                 │   │
│  │                          CoreML                    │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  ONNX 优点:                                                 │
│  - 跨框架兼容: 一次导出，多处部署                          │
│  - 丰富的推理引擎支持                                      │
│  - 图优化机会 (算子融合、常量折叠)                         │
│                                                             │
│  ONNX 导出参数:                                             │
│  - input_names: 输入节点名称                               │
│  - output_names: 输出节点名称                              │
│  - dynamic_axes: 动态维度 (如可变 batch size)              │
│  - opset_version: ONNX 算子集版本                          │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 检查 ONNX 是否可用
# ============================================================
print("=" * 60)
print("ONNX 环境检查")
print("=" * 60)

try:
    import onnx
    ONNX_AVAILABLE = True
    print(f"\n✓ ONNX 已安装，版本: {onnx.__version__}")
except ImportError:
    ONNX_AVAILABLE = False
    print("\n✗ ONNX 未安装")
    print("  安装命令: pip install onnx")

try:
    import onnxruntime as ort
    ORT_AVAILABLE = True
    print(f"✓ ONNX Runtime 已安装，版本: {ort.__version__}")
except ImportError:
    ORT_AVAILABLE = False
    print("✗ ONNX Runtime 未安装")
    print("  安装命令: pip install onnxruntime")

In [ ]:
# ============================================================
# ONNX 导出
# ============================================================
if ONNX_AVAILABLE:
    print("=" * 60)
    print("ONNX 导出")
    print("=" * 60)
    
    with tempfile.TemporaryDirectory() as tmpdir:
        onnx_path = os.path.join(tmpdir, "model.onnx")
        
        # 导出为 ONNX 格式
        torch.onnx.export(
            model,                          # 模型
            dummy_input,                    # 示例输入
            onnx_path,                      # 输出路径
            input_names=['input'],          # 输入节点名称
            output_names=['output'],        # 输出节点名称
            dynamic_axes={                  # 动态维度配置
                'input': {0: 'batch_size'},   # batch 维度可变
                'output': {0: 'batch_size'}
            },
            opset_version=14,               # ONNX 算子集版本
            do_constant_folding=True        # 常量折叠优化
        )
        
        # 验证导出的模型
        onnx_model = onnx.load(onnx_path)
        onnx.checker.check_model(onnx_model)
        
        print(f"\n✓ ONNX 导出成功!")
        print(f"  文件路径: {onnx_path}")
        print(f"  文件大小: {os.path.getsize(onnx_path) / (1024*1024):.2f} MB")
        print(f"  Opset 版本: {onnx_model.opset_import[0].version}")
        
        # 查看模型输入输出
        print(f"\n模型输入:")
        for inp in onnx_model.graph.input:
            print(f"  {inp.name}: {[d.dim_value or d.dim_param for d in inp.type.tensor_type.shape.dim]}")
        print(f"\n模型输出:")
        for out in onnx_model.graph.output:
            print(f"  {out.name}: {[d.dim_value or d.dim_param for d in out.type.tensor_type.shape.dim]}")
else:
    print("跳过 ONNX 导出示例 (ONNX 未安装)")

In [ ]:
# ============================================================
# 使用 ONNX Runtime 推理
# ============================================================
if ONNX_AVAILABLE and ORT_AVAILABLE:
    print("=" * 60)
    print("ONNX Runtime 推理")
    print("=" * 60)
    
    with tempfile.TemporaryDirectory() as tmpdir:
        onnx_path = os.path.join(tmpdir, "model.onnx")
        
        # 导出模型
        torch.onnx.export(model, dummy_input, onnx_path, opset_version=14)
        
        # 创建 ONNX Runtime 推理会话
        session = ort.InferenceSession(onnx_path)
        
        # 获取输入名称
        input_name = session.get_inputs()[0].name
        
        # 使用 ONNX Runtime 推理
        ort_output = session.run(None, {input_name: dummy_input.numpy()})[0]
        
        # 与 PyTorch 输出比较
        with torch.no_grad():
            torch_output = model(dummy_input).numpy()
        
        diff = abs(torch_output - ort_output).max()
        
        print(f"\n推理结果比较:")
        print(f"  PyTorch 输出形状: {torch_output.shape}")
        print(f"  ONNX Runtime 输出形状: {ort_output.shape}")
        print(f"  最大差异: {diff:.6f}")
        print(f"  输出一致: {diff < 1e-5}")
else:
    print("跳过 ONNX Runtime 推理示例")

<cell_type>markdown</cell_type>## 5. 导出最佳实践

**核心概念**: 导出前的检查和验证是确保模型正确部署的关键

```
┌─────────────────────────────────────────────────────────────┐
│                   导出前检查清单                             │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  1. 模型状态检查                                            │
│     ├── [✓] 模型处于 eval 模式                             │
│     ├── [✓] BatchNorm/Dropout 行为正确                     │
│     └── [✓] 无 NaN/Inf 参数                                │
│                                                             │
│  2. 输入输出检查                                            │
│     ├── [✓] 输入形状正确                                   │
│     ├── [✓] 输出形状正确                                   │
│     └── [✓] 数据类型匹配                                   │
│                                                             │
│  3. 导出后验证                                              │
│     ├── [✓] 导出模型可加载                                 │
│     ├── [✓] 输出与原模型一致                               │
│     └── [✓] 多个测试样本验证                               │
│                                                             │
│  常见问题:                                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  问题: 导出后输出不一致                             │   │
│  │  原因: 模型未设置 eval 模式，Dropout 仍在工作       │   │
│  │  解决: model.eval()                                 │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 导出前检查工具
# ============================================================

def pre_export_checklist(model, dummy_input):
    """
    导出前检查清单
    
    在导出模型之前，运行此函数确保模型状态正确
    
    参数:
        model: PyTorch 模型
        dummy_input: 示例输入
        
    返回:
        bool: 是否通过所有检查
    """
    print("=" * 60)
    print("导出前检查清单")
    print("=" * 60)
    
    all_passed = True
    
    # 1. 检查模型模式
    is_eval = not model.training
    status = "✓" if is_eval else "✗"
    print(f"\n[{status}] 模型处于 eval 模式")
    if not is_eval:
        print("    警告: 请调用 model.eval() 设置为评估模式")
        all_passed = False
    
    # 2. 前向传播测试
    try:
        with torch.no_grad():
            _ = model(dummy_input)
        print("[✓] 前向传播正常")
    except Exception as e:
        print(f"[✗] 前向传播失败: {e}")
        all_passed = False
    
    # 3. 输入形状信息
    print(f"[i] 输入形状: {list(dummy_input.shape)}")
    print(f"[i] 输入数据类型: {dummy_input.dtype}")
    
    # 4. 参数统计
    total_params = sum(p.numel() for p in model.parameters())
    print(f"[i] 总参数量: {total_params:,}")
    
    # 5. 检查 NaN/Inf 参数
    has_nan = any(torch.isnan(p).any() for p in model.parameters())
    has_inf = any(torch.isinf(p).any() for p in model.parameters())
    
    status = "✗" if has_nan else "✓"
    print(f"[{status}] 无 NaN 参数")
    if has_nan:
        all_passed = False
    
    status = "✗" if has_inf else "✓"
    print(f"[{status}] 无 Inf 参数")
    if has_inf:
        all_passed = False
    
    # 6. 检查是否有未初始化的参数
    uninitialized = []
    for name, param in model.named_parameters():
        if param.abs().sum() == 0:
            uninitialized.append(name)
    
    if uninitialized:
        print(f"[!] 警告: 以下参数可能未初始化: {uninitialized[:3]}...")
    
    print("\n" + "=" * 60)
    return all_passed


# 运行检查
model.eval()
ready = pre_export_checklist(model, dummy_input)
print(f"\n准备导出: {'是 ✓' if ready else '否 ✗'}")

In [ ]:
# ============================================================
# 模型输出比较工具
# ============================================================

def compare_model_outputs(model1, model2, test_inputs, atol=1e-5, rtol=1e-5):
    """
    比较两个模型的输出
    
    用于验证导出后的模型与原始模型输出一致
    
    参数:
        model1: 原始模型
        model2: 导出后的模型
        test_inputs: 测试输入列表
        atol: 绝对误差容忍度
        rtol: 相对误差容忍度
        
    返回:
        dict: 比较结果
    """
    all_match = True
    max_diff = 0.0
    total_diff = 0.0
    
    for inp in test_inputs:
        with torch.no_grad():
            out1 = model1(inp)
            out2 = model2(inp)
        
        # 计算差异
        diff = (out1 - out2).abs()
        max_diff = max(max_diff, diff.max().item())
        total_diff += diff.mean().item()
        
        # 检查是否匹配
        if not torch.allclose(out1, out2, atol=atol, rtol=rtol):
            all_match = False
    
    return {
        'num_tests': len(test_inputs),
        'all_match': all_match,
        'max_diff': max_diff,
        'mean_diff': total_diff / len(test_inputs)
    }


# 比较原始模型和 TorchScript 模型
print("=" * 60)
print("模型输出比较")
print("=" * 60)

test_inputs = [torch.randn(1, 3, 32, 32) for _ in range(10)]

results = compare_model_outputs(
    model, traced_model, test_inputs,
    atol=1e-5, rtol=1e-5
)

print(f"\n比较结果:")
print(f"  测试样本数: {results['num_tests']}")
print(f"  全部匹配: {results['all_match']}")
print(f"  最大差异: {results['max_diff']:.6f}")
print(f"  平均差异: {results['mean_diff']:.6f}")

<cell_type>markdown</cell_type>## 6. 导出格式对比

```
┌─────────────────────────────────────────────────────────────┐
│                   导出格式对比                               │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  格式        优点                缺点              适用场景  │
│  ────        ────                ────              ────────  │
│  TorchScript PyTorch原生         仅PyTorch生态     PyTorch部署│
│              支持动态控制流      部分算子不支持              │
│                                                             │
│  ONNX        跨框架通用          部分算子不支持    通用部署  │
│              生态丰富            动态形状支持有限            │
│                                                             │
│  TensorRT    NVIDIA GPU极致优化  仅NVIDIA GPU      GPU推理   │
│              INT8/FP16加速       需要重新编译                │
│                                                             │
│  CoreML      Apple设备优化       仅Apple设备       iOS/macOS │
│              Neural Engine加速                               │
│                                                             │
└─────────────────────────────────────────────────────────────┘

选择建议:
┌─────────────────────────────────────────────────────────────┐
│  场景                          推荐格式                     │
│  ────                          ────────                     │
│  PyTorch 服务器部署            TorchScript                  │
│  跨框架/多平台部署             ONNX                         │
│  NVIDIA GPU 高性能推理         TensorRT                     │
│  Apple 设备部署                CoreML                       │
│  边缘设备/嵌入式               ONNX + 专用推理引擎          │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 性能对比基准测试
# ============================================================

def benchmark(model_fn, input_tensor, num_runs=100, name="Model"):
    """
    模型推理基准测试
    
    参数:
        model_fn: 模型或可调用对象
        input_tensor: 输入张量
        num_runs: 测试次数
        name: 模型名称
        
    返回:
        dict: 性能统计
    """
    # 预热 (让 CPU 缓存稳定)
    for _ in range(10):
        with torch.no_grad():
            _ = model_fn(input_tensor)
    
    # 计时
    times = []
    for _ in range(num_runs):
        start = time.perf_counter()
        with torch.no_grad():
            _ = model_fn(input_tensor)
        times.append((time.perf_counter() - start) * 1000)
    
    times = np.array(times)
    
    return {
        'name': name,
        'mean': times.mean(),
        'std': times.std(),
        'min': times.min(),
        'max': times.max()
    }


# 性能对比测试
print("=" * 60)
print("推理性能对比")
print("=" * 60)

test_input = torch.randn(1, 3, 32, 32)

results = [
    benchmark(model, test_input, name="PyTorch (eager)"),
    benchmark(traced_model, test_input, name="TorchScript (trace)"),
    benchmark(scripted_model, test_input, name="TorchScript (script)")
]

print(f"\n{'模型':<25} {'平均(ms)':<12} {'标准差':<10} {'最小':<10} {'最大':<10}")
print("-" * 67)
for r in results:
    print(f"{r['name']:<25} {r['mean']:<12.3f} {r['std']:<10.3f} {r['min']:<10.3f} {r['max']:<10.3f}")

# 计算加速比
baseline = results[0]['mean']
print(f"\n加速比 (相对于 PyTorch eager):")
for r in results[1:]:
    speedup = baseline / r['mean']
    print(f"  {r['name']}: {speedup:.2f}x")

<cell_type>markdown</cell_type>## 总结

本教程介绍了模型导出的核心概念和最佳实践：

### 核心知识点

| 主题 | 关键内容 |
|:-----|:---------|
| 模型分析 | 参数统计、大小估算、性能分析 |
| TorchScript | Tracing (静态图) vs Scripting (动态控制流) |
| ONNX | 跨框架格式、动态维度、算子集版本 |
| 验证 | 输出一致性检查、多样本测试 |

### 导出检查清单

```
导出前:
✓ 模型设置为 eval 模式
✓ 前向传播测试通过
✓ 无 NaN/Inf 参数
✓ 输入形状正确

导出后:
✓ 模型可正常加载
✓ 输出与原模型一致
✓ 多个测试样本验证通过
✓ 性能符合预期
```

### PyTorch 导出 API 速查

```python
# TorchScript Tracing (静态模型)
traced = torch.jit.trace(model, example_input)
traced.save("model.pt")

# TorchScript Scripting (动态控制流)
scripted = torch.jit.script(model)
scripted.save("model.pt")

# 加载 TorchScript 模型
loaded = torch.jit.load("model.pt")

# ONNX 导出
torch.onnx.export(
    model, example_input, "model.onnx",
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch'}},
    opset_version=14
)
```

### 下一步学习

- **05_Advanced_Optimization_tutorial.ipynb**: 高级优化技术 (混合精度、算子融合等)